In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import time

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    make_scorer,
    matthews_corrcoef,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import pairwise_distances
from sklearn.utils.validation import check_is_fitted

from sklearn_extra.cluster import KMedoids

import sys
sys.path.append("../../utils/")

from utils import *

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

NOMBRE_EXPERIMENTO = "BCCC17__split__v1__kmedoids_per_class__v1"
CARPETA_DATASET = "BCCC17__split__v1"

NOMBRE_DATASET_TRAIN = f"{CARPETA_DATASET}__train.csv"
NOMBRE_DATASET_TEST = f"{CARPETA_DATASET}__test.csv"

RUTA_DATASET = PROJECT_ROOT / "02_datasets" / "processed" / CARPETA_DATASET
RUTA_RESULTADOS = PROJECT_ROOT / "04_experimentos" / "logs" / "resultados" / NOMBRE_EXPERIMENTO

NOMBRE_RESULTADOS_CV_CSV = f"{NOMBRE_EXPERIMENTO}__folds.csv"
NOMBRE_RESULTADOS_CV_JSON = f"{NOMBRE_EXPERIMENTO}__summary_cv.json"
NOMBRE_RESULTADOS_TEST_JSON = f"{NOMBRE_EXPERIMENTO}__summary_test.json"
NOMBRE_RESULTADOS_TEST_CSV = f"{NOMBRE_EXPERIMENTO}__metricas_test.csv"
NOMBRE_RESULTADOS_TEST_CM_CSV = f"{NOMBRE_EXPERIMENTO}__confusion_matrix_test.csv"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"

N_SPLITS = 5
SHUFFLE = True
RANDOM_STATE = 42

# ===== CONFIG KMEDOIDS POR CLASE =====
MAX_SAMPLES_PER_CLASS = 10000
K_MEDOIDS_PER_CLASS = 10
METRIC = "euclidean"

# Para calcular pseudo-probabilidades desde distancias
DISTANCE_TO_PROBA_EPS = 1e-9

In [3]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Ruta dataset train:")
print((RUTA_DATASET / NOMBRE_DATASET_TRAIN).resolve())
print()

print("Ruta dataset test:")
print((RUTA_DATASET / NOMBRE_DATASET_TEST).resolve())
print()

print("Ruta resultados:")
print(RUTA_RESULTADOS.resolve())

Ruta dataset train:
/home/javier/TFG_MODELOS_MEDOIDES/02_datasets/processed/BCCC17__split__v1/BCCC17__split__v1__train.csv

Ruta dataset test:
/home/javier/TFG_MODELOS_MEDOIDES/02_datasets/processed/BCCC17__split__v1/BCCC17__split__v1__test.csv

Ruta resultados:
/home/javier/TFG_MODELOS_MEDOIDES/04_experimentos/logs/resultados/BCCC17__split__v1__kmedoids_per_class__v1


In [4]:
df_train = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TRAIN,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset train:")
print(df_train.shape)

df_train.head()

Forma del dataset train:
(1890959, 64)


,DST_PORT,PROTOCOL,DURATION,PACKETS_COUNT,FWD_TOTAL_PAYLOAD_BYTES,PAYLOAD_BYTES_MAX,PAYLOAD_BYTES_MIN,PAYLOAD_BYTES_MEAN,PAYLOAD_BYTES_VARIANCE,FWD_PAYLOAD_BYTES_VARIANCE,...,BWD_SYN_FLAG_COUNTS,BWD_CWR_FLAG_COUNTS,BWD_RST_FLAG_COUNTS,PACKETS_IAT_MEAN,FWD_PACKETS_IAT_MEAN,FWD_PACKETS_IAT_STD,BWD_PACKETS_IAT_MEAN,SUBFLOW_FWD_PACKETS,SUBFLOW_FWD_BYTES,LABEL
0,443,0,183.413713,3510,21202,4380,0,988.815100,956633.216239,1625.416188,...,1,0,0,5.226951e-02,0.129978,2.692378,8.745763e-02,353.000000,5300.500000,0
1,80,0,172.747578,54,2631,796,0,137.166667,72709.138889,31099.854935,...,1,0,0,3.259388e+00,6.168708,4.541743,7.196825e+00,1.526316,138.473684,0
2,36739,0,0.000000,1,0,0,0,0.000000,0.000000,0.000000,...,0,0,0,1.499345e+09,0.000000,0.000000,1.499345e+09,0.000000,0.000000,0
3,53,1,0.000274,4,76,121,38,79.500000,1722.250000,0.000000,...,0,0,0,9.139000e-05,0.000001,0.000000,2.150000e-06,0.000000,0.000000,0
4,443,0,6.020985,13,599,517,0,60.153846,19067.360947,35175.138889,...,1,0,0,5.017487e-01,1.199572,2.374952,9.996383e-01,6.000000,599.000000,0


In [5]:
if LABEL_COL not in df_train.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en train")

print("Última columna train:", df_train.columns[-1])
print("Tipo de LABEL train:", df_train[LABEL_COL].dtype)
print()

print("Distribución de clases en train:")
display(df_train[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna train: LABEL
Tipo de LABEL train: int64

Distribución de clases en train:


,count
LABEL,
0,1372038
1,276914
2,129058
3,76583
4,7625
5,6691
6,5485
7,4759
8,4406


In [6]:
X_train = df_train.drop(columns=[LABEL_COL]).copy()
y_train = df_train[LABEL_COL].copy()

print("Shape X_train:", X_train.shape)
print("Shape y_train:", y_train.shape)

Shape X_train: (1890959, 63)
Shape y_train: (1890959,)


In [7]:
columnas_no_numericas_train = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_train:")
print(columnas_no_numericas_train)

if len(columnas_no_numericas_train) > 0:
    raise ValueError("Hay columnas no numéricas en X_train. Revísalas antes de seguir.")

Columnas no numéricas en X_train:
[]


In [8]:
pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("kmedoids_per_class", KMedoidsPerClassClassifier(
        k_medoids_per_class=K_MEDOIDS_PER_CLASS,
        metric=METRIC,
        random_state=RANDOM_STATE,
        distance_to_proba_eps=DISTANCE_TO_PROBA_EPS,
        max_samples_per_class=MAX_SAMPLES_PER_CLASS
    ))
])

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('kmedoids_per_class', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"with_centering with_centering: bool, default=TrueIf `True`, center the data before scaling.This will cause :meth:`transform` to raise an exception when attemptedon sparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_scaling with_scaling: bool, default=TrueIf `True`, scale the data to interquartile range.",True
,"quantile_range quantile_range: tuple (q_min, q_max), 0.0 < q_min < q_max < 100.0, default=(25.0, 75.0)Quantile range used to calculate `scale_`. By default this is equal tothe IQR, i.e., `q_min` is the first quantile and `q_max` is the thirdquantile... versionadded:: 0.18","(25.0, ...)"
,"copy copy: bool, default=TrueIf `False`, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"unit_variance unit_variance: bool, default=FalseIf `True`, scale data so that normally distributed features have avariance of 1. In general, if the difference between the x-values of`q_max` and `q_min` for a standard normal distribution is greaterthan 1, the dataset will be scaled down. If less than 1, the datasetwill be scaled up... versionadded:: 0.24",False
,k_medoids_per_class,10
,metric,'euclidean'


In [9]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE
)

cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [10]:
scoring = {
    "accuracy": "accuracy",

    "precision_weighted": "precision_weighted",
    "recall_weighted": "recall_weighted",
    "f1_weighted": "f1_weighted",

    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",

    "mcc": make_scorer(matthews_corrcoef),

    "roc_auc": "roc_auc_ovr_weighted",
}

In [11]:
cv_results = cross_validate(
    estimator=pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1
)

cv_results.keys()

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn_extra/cluster/_k_medoids.py:329: UserWarning: Cluster 5 is empty! self.labels_[self.medoid_indices_[5]] may not be labeled with its corresponding cluster (5).
  warnings.warn(
/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn_extra/cluster/_k_medoids.py:329: UserWarning: Cluster 6 is empty! self.labels_[self.medoid_indices_[6]] may not be labeled with its corresponding cluster (6).
  warnings.warn(
/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn_extra/cluster/_k_medoids.py:329: UserWarning: Cluster 3 is empty! self.labels_[self.medoid_indices_[3]] may not be labeled with its corresponding cluster (3).
  warnings.warn(
/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn_extra/cluster/_k_medoids.py:329: UserWarning: Cluster 4 is empty! self.labels_[self.medoid_indices_[4]] may not be labeled with its corresponding cluster (4).
  warnings.warn(


/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
             ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 96, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/utils/_response.py", line 235, in _get_response_values
    raise ValueError(
ValueError: Pipeline should ei

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn_extra/cluster/_k_medoids.py:329: UserWarning: Cluster 8 is empty! self.labels_[self.medoid_indices_[8]] may not be labeled with its corresponding cluster (8).
  warnings.warn(
/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn_extra/cluster/_k_medoids.py:329: UserWarning: Cluster 9 is empty! self.labels_[self.medoid_indices_[9]] may not be labeled with its corresponding cluster (9).
  warnings.warn(


/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn_extra/cluster/_k_medoids.py:329: UserWarning: Cluster 3 is empty! self.labels_[self.medoid_indices_[3]] may not be labeled with its corresponding cluster (3).
  warnings.warn(
/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn_extra/cluster/_k_medoids.py:329: UserWarning: Cluster 4 is empty! self.labels_[self.medoid_indices_[4]] may not be labeled with its corresponding cluster (4).
  warnings.warn(


/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
             ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 96, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/utils/_response.py", line 235, in _get_response_values
    raise ValueError(
ValueError: Pipeline should ei

...

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
             ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 96, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/utils/_response.py", line 235, in _get_response_values
    raise ValueError(
ValueError: Pipeline should ei

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn_extra/cluster/_k_medoids.py:329: UserWarning: Cluster 3 is empty! self.labels_[self.medoid_indices_[3]] may not be labeled with its corresponding cluster (3).
  warnings.warn(


/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
             ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 96, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/utils/_response.py", line 235, in _get_response_values
    raise ValueError(
ValueError: Pipeline should ei

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
             ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 96, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/utils/_response.py", line 235, in _get_response_values
    raise ValueError(
ValueError: Pipeline should ei

dict_keys(['fit_time', 'score_time', 'test_accuracy', 'test_precision_weighted', 'test_recall_weighted', 'test_f1_weighted', 'test_precision_macro', 'test_recall_macro', 'test_f1_macro', 'test_mcc', 'test_roc_auc'])

In [12]:
df_folds = pd.DataFrame({
    "fold": np.arange(1, N_SPLITS + 1),

    "accuracy": cv_results["test_accuracy"],

    "precision_weighted": cv_results["test_precision_weighted"],
    "recall_weighted": cv_results["test_recall_weighted"],
    "f1_weighted": cv_results["test_f1_weighted"],

    "precision_macro": cv_results["test_precision_macro"],
    "recall_macro": cv_results["test_recall_macro"],
    "f1_macro": cv_results["test_f1_macro"],

    "mcc": cv_results["test_mcc"],

    "roc_auc": cv_results["test_roc_auc"],

    "fit_time": cv_results["fit_time"],
    "score_time": cv_results["score_time"]
})

df_folds

,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,0.619654,0.903280,0.619654,0.707527,0.257253,0.772547,0.281792,0.499485,NaN,15.259993,0.415197
1,2,0.549747,0.896587,0.549747,0.639156,0.262816,0.811680,0.282446,0.462654,NaN,13.886598,0.419636
2,3,0.641423,0.893267,0.641423,0.720006,0.265590,0.806078,0.296714,0.515996,NaN,13.026891,0.580638
3,4,0.545014,0.899360,0.545014,0.639351,0.267203,0.813193,0.277120,0.454102,NaN,13.907274,0.641732
4,5,0.546277,0.901630,0.546277,0.641531,0.266894,0.814129,0.277448,0.455803,NaN,16.086197,0.609403


In [13]:
summary_cv = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_train": str(RUTA_DATASET / NOMBRE_DATASET_TRAIN),
    "shape_train": {
        "rows": int(df_train.shape[0]),
        "cols": int(df_train.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_splits": N_SPLITS,
        "shuffle": SHUFFLE,
        "random_state": RANDOM_STATE,
        "scaler": "RobustScaler",
        "modelo": "KMedoidsPerClassClassifier",
        "k_medoids_per_class": K_MEDOIDS_PER_CLASS,
        "metric": METRIC
    },
    "metricas_media": {
        "accuracy": float(df_folds["accuracy"].mean()),

        "precision_weighted": float(df_folds["precision_weighted"].mean()),
        "recall_weighted": float(df_folds["recall_weighted"].mean()),
        "f1_weighted": float(df_folds["f1_weighted"].mean()),

        "precision_macro": float(df_folds["precision_macro"].mean()),
        "recall_macro": float(df_folds["recall_macro"].mean()),
        "f1_macro": float(df_folds["f1_macro"].mean()),

        "mcc": float(df_folds["mcc"].mean()),
        "roc_auc": float(df_folds["roc_auc"].mean()),
        "fit_time": float(df_folds["fit_time"].mean()),
        "score_time": float(df_folds["score_time"].mean())
    },
    "metricas_std": {
        "accuracy": float(df_folds["accuracy"].std(ddof=1)),

        "precision_weighted": float(df_folds["precision_weighted"].std(ddof=1)),
        "recall_weighted": float(df_folds["recall_weighted"].std(ddof=1)),
        "f1_weighted": float(df_folds["f1_weighted"].std(ddof=1)),

        "precision_macro": float(df_folds["precision_macro"].std(ddof=1)),
        "recall_macro": float(df_folds["recall_macro"].std(ddof=1)),
        "f1_macro": float(df_folds["f1_macro"].std(ddof=1)),

        "mcc": float(df_folds["mcc"].std(ddof=1)),
        "roc_auc": float(df_folds["roc_auc"].std(ddof=1)),
        "fit_time": float(df_folds["fit_time"].std(ddof=1)),
        "score_time": float(df_folds["score_time"].std(ddof=1))
    }
}

summary_cv

{'experimento': 'BCCC17__split__v1__kmedoids_per_class__v1',
 'dataset_train': '/home/javier/TFG_MODELOS_MEDOIDES/02_datasets/processed/BCCC17__split__v1/BCCC17__split__v1__train.csv',
 'shape_train': {'rows': 1890959, 'cols': 64},
 'parametros': {'label_col': 'LABEL',
  'n_splits': 5,
  'shuffle': True,
  'random_state': 42,
  'scaler': 'RobustScaler',
  'modelo': 'KMedoidsPerClassClassifier',
  'k_medoids_per_class': 10,
  'metric': 'euclidean'},
 'metricas_media': {'accuracy': 0.580422931355968,
  'precision_weighted': 0.8988246311327244,
  'recall_weighted': 0.580422931355968,
  'f1_weighted': 0.6695141167834915,
  'precision_macro': 0.26395125430539246,
  'recall_macro': 0.8035255703706511,
  'f1_macro': 0.2831039510639089,
  'mcc': 0.47760815881993707,
  'roc_auc': nan,
  'fit_time': 14.433390474319458,
  'score_time': 0.5333210945129394},
 'metricas_std': {'accuracy': 0.046424033881404966,
  'precision_weighted': 0.003996931933689966,
  'recall_weighted': 0.046424033881404966,
 

In [14]:
print("========== RESULTADOS CV ==========")
print(f"Accuracy            : {summary_cv['metricas_media']['accuracy']:.6f} ± {summary_cv['metricas_std']['accuracy']:.6f}")
print()

print(f"Precision weighted  : {summary_cv['metricas_media']['precision_weighted']:.6f} ± {summary_cv['metricas_std']['precision_weighted']:.6f}")
print(f"Recall weighted     : {summary_cv['metricas_media']['recall_weighted']:.6f} ± {summary_cv['metricas_std']['recall_weighted']:.6f}")
print(f"F1 weighted         : {summary_cv['metricas_media']['f1_weighted']:.6f} ± {summary_cv['metricas_std']['f1_weighted']:.6f}")
print()

print(f"Precision macro     : {summary_cv['metricas_media']['precision_macro']:.6f} ± {summary_cv['metricas_std']['precision_macro']:.6f}")
print(f"Recall macro        : {summary_cv['metricas_media']['recall_macro']:.6f} ± {summary_cv['metricas_std']['recall_macro']:.6f}")
print(f"F1 macro            : {summary_cv['metricas_media']['f1_macro']:.6f} ± {summary_cv['metricas_std']['f1_macro']:.6f}")
print()

print(f"MCC                 : {summary_cv['metricas_media']['mcc']:.6f} ± {summary_cv['metricas_std']['mcc']:.6f}")
print(f"ROC AUC             : {summary_cv['metricas_media']['roc_auc']:.6f} ± {summary_cv['metricas_std']['roc_auc']:.6f}")
print()
print(f"Fit time medio      : {summary_cv['metricas_media']['fit_time']:.6f}")
print(f"Score time medio    : {summary_cv['metricas_media']['score_time']:.6f}")

========== RESULTADOS CV ==========
Accuracy            : 0.580423 ± 0.046424

Precision weighted  : 0.898825 ± 0.003997
Recall weighted     : 0.580423 ± 0.046424
F1 weighted         : 0.669514 ± 0.040648

Precision macro     : 0.263951 ± 0.004126
Recall macro        : 0.803526 ± 0.017597
F1 macro            : 0.283104 ± 0.007987

MCC                 : 0.477608 ± 0.028301
ROC AUC             : nan ± nan

Fit time medio      : 14.433390
Score time medio    : 0.533321


In [15]:
ruta_cv_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_CSV
df_folds.to_csv(ruta_cv_csv, index=False)

print("Resultados por fold guardados en:")
print(ruta_cv_csv.resolve())

Resultados por fold guardados en:
/home/javier/TFG_MODELOS_MEDOIDES/04_experimentos/logs/resultados/BCCC17__split__v1__kmedoids_per_class__v1/BCCC17__split__v1__kmedoids_per_class__v1__folds.csv


In [16]:
ruta_cv_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_JSON

with open(ruta_cv_json, "w", encoding="utf-8") as f:
    json.dump(summary_cv, f, indent=4, ensure_ascii=False)

print("Resumen CV guardado en:")
print(ruta_cv_json.resolve())

Resumen CV guardado en:
/home/javier/TFG_MODELOS_MEDOIDES/04_experimentos/logs/resultados/BCCC17__split__v1__kmedoids_per_class__v1/BCCC17__split__v1__kmedoids_per_class__v1__summary_cv.json


In [17]:
df_folds

,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,0.619654,0.903280,0.619654,0.707527,0.257253,0.772547,0.281792,0.499485,NaN,15.259993,0.415197
1,2,0.549747,0.896587,0.549747,0.639156,0.262816,0.811680,0.282446,0.462654,NaN,13.886598,0.419636
2,3,0.641423,0.893267,0.641423,0.720006,0.265590,0.806078,0.296714,0.515996,NaN,13.026891,0.580638
3,4,0.545014,0.899360,0.545014,0.639351,0.267203,0.813193,0.277120,0.454102,NaN,13.907274,0.641732
4,5,0.546277,0.901630,0.546277,0.641531,0.266894,0.814129,0.277448,0.455803,NaN,16.086197,0.609403


In [18]:
df_test = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TEST,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset test:")
print(df_test.shape)

df_test.head()

Forma del dataset test:
(472740, 64)


,DST_PORT,PROTOCOL,DURATION,PACKETS_COUNT,FWD_TOTAL_PAYLOAD_BYTES,PAYLOAD_BYTES_MAX,PAYLOAD_BYTES_MIN,PAYLOAD_BYTES_MEAN,PAYLOAD_BYTES_VARIANCE,FWD_PAYLOAD_BYTES_VARIANCE,...,BWD_SYN_FLAG_COUNTS,BWD_CWR_FLAG_COUNTS,BWD_RST_FLAG_COUNTS,PACKETS_IAT_MEAN,FWD_PACKETS_IAT_MEAN,FWD_PACKETS_IAT_STD,BWD_PACKETS_IAT_MEAN,SUBFLOW_FWD_PACKETS,SUBFLOW_FWD_BYTES,LABEL
0,443,0,0.000000,1,0,0,0,0.0,0.00,0.0,...,0,0,0,1.499195e+09,1.499195e+09,0.0,0.000000e+00,0.0,0.0,0
1,53,1,0.741115,4,62,61,31,46.0,225.00,0.0,...,0,0,0,2.470384e-01,2.860000e-06,0.0,4.721000e-05,0.0,0.0,0
2,53,1,0.024214,4,82,124,41,82.5,1722.25,0.0,...,0,0,0,8.071340e-03,3.100000e-06,0.0,3.100000e-06,0.0,0.0,0
3,3737,0,0.000048,2,0,0,0,0.0,0.00,0.0,...,0,0,1,4.816000e-05,1.499451e+09,0.0,1.499451e+09,0.0,0.0,2
4,389,0,0.000048,2,0,0,0,0.0,0.00,0.0,...,0,0,0,4.792000e-05,1.499436e+09,0.0,1.499436e+09,0.0,0.0,0


In [19]:
if LABEL_COL not in df_test.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en test")

print("Última columna test:", df_test.columns[-1])
print("Tipo de LABEL test:", df_test[LABEL_COL].dtype)
print()

print("Distribución de clases en test:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna test: LABEL
Tipo de LABEL test: int64

Distribución de clases en test:


,count
LABEL,
0,343010
1,69229
2,32265
3,19146
4,1906
5,1673
6,1371
7,1190
8,1102


In [20]:
X_test = df_test.drop(columns=[LABEL_COL]).copy()
y_test = df_test[LABEL_COL].copy()

print("Shape X_test:", X_test.shape)
print("Shape y_test:", y_test.shape)

Shape X_test: (472740, 63)
Shape y_test: (472740,)


In [21]:
columnas_no_numericas_test = X_test.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_test:")
print(columnas_no_numericas_test)

if len(columnas_no_numericas_test) > 0:
    raise ValueError("Hay columnas no numéricas en X_test. Revísalas antes de seguir.")

Columnas no numéricas en X_test:
[]


In [22]:
inicio = time.time()

pipeline.fit(X_train, y_train)

fin = time.time()

print("Modelo final entrenado con todo el dataset train.")
print(f"Tiempo de entrenamiento: {fin - inicio:.2f} segundos")

Modelo final entrenado con todo el dataset train.
Tiempo de entrenamiento: 18.53 segundos


/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn_extra/cluster/_k_medoids.py:329: UserWarning: Cluster 9 is empty! self.labels_[self.medoid_indices_[9]] may not be labeled with its corresponding cluster (9).
  warnings.warn(


In [23]:
y_pred_test = pipeline.predict(X_test)

print("Predicciones en test generadas.")
print("Número de predicciones:", len(y_pred_test))

y_proba_test = pipeline.predict_proba(X_test)

roc_auc_test = roc_auc_score(
    y_test,
    y_proba_test,
    multi_class="ovr",
    average="weighted",
    labels=pipeline.named_steps["kmedoids_per_class"].classes_
)

Predicciones en test generadas.
Número de predicciones: 472740


In [24]:
metricas_test = {
    "accuracy": accuracy_score(y_test, y_pred_test),

    "precision_weighted": precision_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred_test, average="weighted", zero_division=0),

    "precision_macro": precision_score(y_test, y_pred_test, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred_test, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred_test, average="macro", zero_division=0),

    "roc_auc": roc_auc_test,

    "mcc": matthews_corrcoef(y_test, y_pred_test)
}

metricas_test

{'accuracy': 0.6223463214451919,
 'precision_weighted': 0.9091423681278232,
 'recall_weighted': 0.6223463214451919,
 'f1_weighted': 0.7140111992676395,
 'precision_macro': 0.2808264654346283,
 'recall_macro': 0.7335719831556153,
 'f1_macro': 0.3013860852547036,
 'roc_auc': 0.8607244277400775,
 'mcc': 0.5043941842624401}

In [25]:
print("========== RESULTADOS TEST ==========")
print(f"Accuracy            : {metricas_test['accuracy']:.6f}")
print()

print(f"Precision weighted  : {metricas_test['precision_weighted']:.6f}")
print(f"Recall weighted     : {metricas_test['recall_weighted']:.6f}")
print(f"F1 weighted         : {metricas_test['f1_weighted']:.6f}")
print()

print(f"Precision macro     : {metricas_test['precision_macro']:.6f}")
print(f"Recall macro        : {metricas_test['recall_macro']:.6f}")
print(f"F1 macro            : {metricas_test['f1_macro']:.6f}")
print()

print(f"MCC                 : {metricas_test['mcc']:.6f}")
print(f"ROC AUC             : {metricas_test['roc_auc']:.6f}")

========== RESULTADOS TEST ==========
Accuracy            : 0.622346

Precision weighted  : 0.909142
Recall weighted     : 0.622346
F1 weighted         : 0.714011

Precision macro     : 0.280826
Recall macro        : 0.733572
F1 macro            : 0.301386

MCC                 : 0.504394
ROC AUC             : 0.860724


In [26]:
labels_ordenadas = sorted(pd.unique(pd.concat([y_test, pd.Series(y_pred_test)])))

cm = confusion_matrix(y_test, y_pred_test, labels=labels_ordenadas)
df_cm = pd.DataFrame(cm, index=labels_ordenadas, columns=labels_ordenadas)

print("Matriz de confusión en test:")
display(df_cm)

Matriz de confusión en test:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,186437,18942,2350,22,32226,22559,7758,11000,12611,3741,5325,14711,16671,8657
1,3222,52815,1,6296,0,572,916,4396,83,180,0,0,739,9
2,21,8,31523,0,0,3,0,1,283,0,4,266,156,0
3,194,2328,0,15936,0,72,486,0,0,67,15,12,0,36
4,301,5,10,0,1589,0,1,0,0,0,0,0,0,0
5,8,99,0,1,1,1354,23,98,0,20,0,0,48,21
6,54,33,0,0,0,111,979,107,0,77,0,1,7,2
7,9,2,0,0,7,12,0,1160,0,0,0,0,0,0
8,6,0,0,0,0,9,0,0,908,0,0,146,0,33
9,5,2,0,0,0,18,27,56,31,884,0,0,1,0


In [27]:
print("========== CLASSIFICATION REPORT TEST ==========")
print(classification_report(y_test, y_pred_test, zero_division=0))

========== CLASSIFICATION REPORT TEST ==========
              precision    recall  f1-score   support

           0       0.98      0.54      0.70    343010
           1       0.71      0.76      0.74     69229
           2       0.93      0.98      0.95     32265
           3       0.72      0.83      0.77     19146
           4       0.05      0.83      0.09      1906
           5       0.05      0.81      0.10      1673
           6       0.10      0.71      0.17      1371
           7       0.07      0.97      0.13      1190
           8       0.07      0.82      0.12      1102
           9       0.18      0.86      0.29      1024
          10       0.07      0.74      0.13       546
          11       0.01      0.79      0.03       271
          12       0.00      0.60      0.00         5
          13       0.00      0.00      0.00         2

    accuracy                           0.62    472740
   macro avg       0.28      0.73      0.30    472740
weighted avg       0.91      0.

In [28]:
summary_test = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_test": str(RUTA_DATASET / NOMBRE_DATASET_TEST),
    "shape_test": {
        "rows": int(df_test.shape[0]),
        "cols": int(df_test.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "scaler": "RobustScaler",
        "modelo": "KMedoidsPerClassClassifier",
        "k_medoids_per_class": K_MEDOIDS_PER_CLASS,
        "metric": METRIC
    },
    "metricas_test": {
        "accuracy": float(metricas_test["accuracy"]),

        "precision_weighted": float(metricas_test["precision_weighted"]),
        "recall_weighted": float(metricas_test["recall_weighted"]),
        "f1_weighted": float(metricas_test["f1_weighted"]),

        "precision_macro": float(metricas_test["precision_macro"]),
        "recall_macro": float(metricas_test["recall_macro"]),
        "f1_macro": float(metricas_test["f1_macro"]),

        "roc_auc": float(metricas_test["roc_auc"]),

        "mcc": float(metricas_test["mcc"])
    }
}

summary_test

{'experimento': 'BCCC17__split__v1__kmedoids_per_class__v1',
 'dataset_test': '/home/javier/TFG_MODELOS_MEDOIDES/02_datasets/processed/BCCC17__split__v1/BCCC17__split__v1__test.csv',
 'shape_test': {'rows': 472740, 'cols': 64},
 'parametros': {'label_col': 'LABEL',
  'scaler': 'RobustScaler',
  'modelo': 'KMedoidsPerClassClassifier',
  'k_medoids_per_class': 10,
  'metric': 'euclidean'},
 'metricas_test': {'accuracy': 0.6223463214451919,
  'precision_weighted': 0.9091423681278232,
  'recall_weighted': 0.6223463214451919,
  'f1_weighted': 0.7140111992676395,
  'precision_macro': 0.2808264654346283,
  'recall_macro': 0.7335719831556153,
  'f1_macro': 0.3013860852547036,
  'roc_auc': 0.8607244277400775,
  'mcc': 0.5043941842624401}}

In [29]:
df_metricas_test = pd.DataFrame([metricas_test])

ruta_test_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CSV
df_metricas_test.to_csv(ruta_test_csv, index=False)

print("Métricas test guardadas en:")
print(ruta_test_csv.resolve())

Métricas test guardadas en:
/home/javier/TFG_MODELOS_MEDOIDES/04_experimentos/logs/resultados/BCCC17__split__v1__kmedoids_per_class__v1/BCCC17__split__v1__kmedoids_per_class__v1__metricas_test.csv


In [30]:
ruta_cm_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CM_CSV
df_cm.to_csv(ruta_cm_csv, index=True)

print("Matriz de confusión test guardada en:")
print(ruta_cm_csv.resolve())

Matriz de confusión test guardada en:
/home/javier/TFG_MODELOS_MEDOIDES/04_experimentos/logs/resultados/BCCC17__split__v1__kmedoids_per_class__v1/BCCC17__split__v1__kmedoids_per_class__v1__confusion_matrix_test.csv


In [31]:
ruta_test_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_JSON

with open(ruta_test_json, "w", encoding="utf-8") as f:
    json.dump(summary_test, f, indent=4, ensure_ascii=False)

print("Resumen test guardado en:")
print(ruta_test_json.resolve())

Resumen test guardado en:
/home/javier/TFG_MODELOS_MEDOIDES/04_experimentos/logs/resultados/BCCC17__split__v1__kmedoids_per_class__v1/BCCC17__split__v1__kmedoids_per_class__v1__summary_test.json


In [32]:
print("========== RESUMEN FINAL ==========")
print("CV:")
print(summary_cv["metricas_media"])
print()
print("TEST:")
print(summary_test["metricas_test"])

========== RESUMEN FINAL ==========
CV:
{'accuracy': 0.580422931355968, 'precision_weighted': 0.8988246311327244, 'recall_weighted': 0.580422931355968, 'f1_weighted': 0.6695141167834915, 'precision_macro': 0.26395125430539246, 'recall_macro': 0.8035255703706511, 'f1_macro': 0.2831039510639089, 'mcc': 0.47760815881993707, 'roc_auc': nan, 'fit_time': 14.433390474319458, 'score_time': 0.5333210945129394}

TEST:
{'accuracy': 0.6223463214451919, 'precision_weighted': 0.9091423681278232, 'recall_weighted': 0.6223463214451919, 'f1_weighted': 0.7140111992676395, 'precision_macro': 0.2808264654346283, 'recall_macro': 0.7335719831556153, 'f1_macro': 0.3013860852547036, 'roc_auc': 0.8607244277400775, 'mcc': 0.5043941842624401}
